# 01 — EDA longitudinal de Kaki / DeepHS Fruit v2

## Proyecto
**Kaki_DeepHS_StorageDays**

## Propósito
Caracterizar, **antes de cualquier modelado**, la señal hiperespectral asociada al tiempo de almacenamiento del caqui y distinguirla de:

- diferencias entre frutos;
- diferencias front/back;
- diferencias entre cámaras;
- estructura longitudinal;
- pérdida progresiva de frutos por medición destructiva;
- posible confusión entre `storage_days` y sesión de adquisición.

## Unidad experimental
- **40 frutos físicos**
- **199 observaciones fruto×día**
- **793 cubos HSI**
- Días: **0, 1, 2, 3, 4, 6, 7, 8**
- Cámaras: **VIS (Specim FX10)** y **VIS_COR (Corning microHSI)**

## Regla principal
Las vistas `front/back` y las dos cámaras **no se tratarán como frutos independientes**.

Este notebook genera espectros representativos por cubo, los agrega a nivel **fruto×día×cámara**, estudia trayectorias longitudinales y exporta un ZIP ligero de revisión.

> **No se entrena ningún modelo predictivo en este notebook.**


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import hashlib
import json
import math
import os
import platform
import re
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "01_kaki_eda_longitudinal"

ROOT = PROJECT_ROOT

RAW = ROOT / "data/raw"
INTERIM = ROOT / "data/interim"
EXTRACTED = INTERIM / "extracted"

MASTER_PATH = INTERIM / "kaki_master_index_v2.csv"

TABLES = ROOT / "tables/exploratory/01_kaki_eda_longitudinal"
FIGURES = ROOT / "figures/exploratory/01_kaki_eda_longitudinal"
RESULTS = ROOT / "results/01_kaki_eda_longitudinal"
EXPORT_ROOT = ROOT / "exports/01_kaki_eda_longitudinal"

for p in [TABLES, FIGURES, RESULTS, EXPORT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Para reproducibilidad de cualquier muestreo de píxeles
RANDOM_SEED = 20260818
rng = np.random.default_rng(RANDOM_SEED)

print("ROOT:", ROOT)
print("MASTER:", MASTER_PATH)


## 2. Cargar índice maestro corregido


In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {MASTER_PATH}. "
        "Ejecute primero la auditoría corregida del notebook 00."
    )

master = pd.read_csv(MASTER_PATH)

required = [
    "fruit_id",
    "storage_days_model",
    "camera_type",
    "side",
    "header_file",
    "data_file",
    "day_code",
]

missing = [c for c in required if c not in master.columns]

if missing:
    raise RuntimeError(f"Faltan columnas requeridas: {missing}")

master["fruit_id"] = pd.to_numeric(master["fruit_id"], errors="raise").astype(int)
master["storage_days_model"] = pd.to_numeric(
    master["storage_days_model"], errors="raise"
).astype(int)

# Rutas físicas: reconstruir de manera robusta
master["header_abs"] = master["header_file"].map(
    lambda x: str(EXTRACTED / str(x))
)
master["data_abs"] = master["data_file"].map(
    lambda x: str(EXTRACTED / str(x))
)

master["header_exists"] = master["header_abs"].map(lambda x: Path(x).exists())
master["data_exists"] = master["data_abs"].map(lambda x: Path(x).exists())

assert len(master) == 793, f"Se esperaban 793 cubos; encontrados {len(master)}"
assert master["fruit_id"].nunique() == 40
assert (
    master[["fruit_id", "storage_days_model"]]
    .drop_duplicates()
    .shape[0]
    == 199
)
assert master["header_exists"].all()
assert master["data_exists"].all()

print("✅ Índice maestro cargado")
print("Cubos:", len(master))
print("Frutos:", master["fruit_id"].nunique())
print(
    "Fruit-days:",
    master[["fruit_id", "storage_days_model"]].drop_duplicates().shape[0]
)
print("Días:", sorted(master["storage_days_model"].unique().tolist()))
print("Cámaras:", master["camera_type"].value_counts().to_dict())


## 3. Estructura experimental básica


In [ ]:
cube_day_counts = (
    master.groupby("storage_days_model")
    .size()
    .reset_index(name="n_cubes")
)

fruit_day_counts = (
    master.groupby("storage_days_model")["fruit_id"]
    .nunique()
    .reset_index(name="n_fruits")
)

view_counts = (
    master.groupby(["fruit_id", "storage_days_model"])
    .agg(
        n_cubes=("header_file", "size"),
        n_cameras=("camera_type", "nunique"),
        n_sides=("side", "nunique"),
    )
    .reset_index()
)

display(cube_day_counts)
display(fruit_day_counts)
display(view_counts["n_cubes"].value_counts().sort_index().rename("n_fruit_days").to_frame())

cube_day_counts.to_csv(TABLES / "01_cube_counts_by_storage_day.csv", index=False)
fruit_day_counts.to_csv(TABLES / "02_fruit_counts_by_storage_day.csv", index=False)
view_counts.to_csv(TABLES / "03_views_per_fruit_day.csv", index=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))

ax.bar(
    fruit_day_counts["storage_days_model"].astype(str),
    fruit_day_counts["n_fruits"]
)

ax.set_xlabel("Días de almacenamiento")
ax.set_ylabel("Número de frutos todavía observados")
ax.set_title("Seguimiento longitudinal y reducción progresiva de frutos")
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig(
    FIGURES / "01_fruits_remaining_by_storage_day.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 4. Sesión de adquisición vs. `storage_days`

Esta tabla es descriptiva y se conservará como alerta metodológica.  
Si cada día de almacenamiento corresponde a una única sesión, el efecto de tiempo y el efecto de sesión no pueden separarse completamente con este experimento.


In [ ]:
session_table = pd.crosstab(
    master["day_code"],
    master["storage_days_model"]
)

display(session_table)

session_summary = (
    master.groupby("day_code")
    .agg(
        n_cubes=("header_file", "size"),
        n_storage_days=("storage_days_model", "nunique"),
        storage_day=("storage_days_model", "first"),
        n_fruits=("fruit_id", "nunique"),
    )
    .reset_index()
    .sort_values("storage_day")
)

display(session_summary)

session_table.to_csv(TABLES / "04_day_code_by_storage_days.csv")
session_summary.to_csv(TABLES / "05_session_summary.csv", index=False)


# 5. Lectura ENVI y extracción del espectro representativo

Los cubos se leen mediante `numpy.memmap`, respetando:

- `samples`
- `lines`
- `bands`
- `data type`
- `interleave`
- `byte order`
- `header offset`

Para cada cubo se crea una máscara del fruto usando un índice de intensidad de tres bandas y umbral de Otsu. Se conserva el componente conectado más grande y se rellenan pequeños huecos.

El espectro representativo de cada cubo será la **mediana por longitud de onda de los píxeles del fruto**. La mediana es menos sensible a brillos especulares y píxeles extremos que la media.

Si una máscara es sospechosa, **se marca para revisión; no se elimina automáticamente**.


In [ ]:
ENVI_DTYPES = {
    1:  np.dtype("u1"),
    2:  np.dtype("i2"),
    3:  np.dtype("i4"),
    4:  np.dtype("f4"),
    5:  np.dtype("f8"),
    6:  np.dtype("c8"),
    9:  np.dtype("c16"),
    12: np.dtype("u2"),
    13: np.dtype("u4"),
    14: np.dtype("i8"),
    15: np.dtype("u8"),
}

def parse_envi_header(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()

    entries = {}
    current_key = None
    current_parts = []

    for raw in lines:
        line = raw.strip()

        if not line or line.upper() == "ENVI":
            continue

        if current_key is not None:
            current_parts.append(line)
            if "}" in line:
                entries[current_key] = " ".join(current_parts)
                current_key = None
                current_parts = []
            continue

        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip().lower()
        value = value.strip()

        if value.startswith("{") and "}" not in value:
            current_key = key
            current_parts = [value]
        else:
            entries[key] = value

    def scalar(key, default=None):
        value = entries.get(key)
        if value is None:
            return default
        return value.strip().strip("{}").strip()

    def integer(key, default=None):
        try:
            return int(float(scalar(key)))
        except Exception:
            return default

    wl = []
    raw_wl = entries.get("wavelength")

    if raw_wl:
        raw_wl = raw_wl.strip().strip("{}")
        for token in re.split(r"[,\s]+", raw_wl):
            token = token.strip()
            if not token:
                continue
            try:
                wl.append(float(token))
            except ValueError:
                pass

    return {
        "samples": integer("samples"),
        "lines": integer("lines"),
        "bands": integer("bands"),
        "header_offset": integer("header offset", 0),
        "data_type": integer("data type"),
        "interleave": scalar("interleave", "").lower(),
        "byte_order": integer("byte order", 0),
        "wavelength_units": scalar("wavelength units"),
        "wavelengths": np.asarray(wl, dtype=float),
    }


def open_envi_cube(header_path, data_path):
    meta = parse_envi_header(header_path)

    for key in ["samples", "lines", "bands", "data_type"]:
        if meta[key] is None:
            raise RuntimeError(f"Header incompleto {header_path}: falta {key}")

    base_dtype = ENVI_DTYPES.get(meta["data_type"])
    if base_dtype is None:
        raise RuntimeError(
            f"ENVI data type no soportado: {meta['data_type']}"
        )

    # byte order: 0 little-endian, 1 big-endian
    if base_dtype.itemsize > 1:
        dtype = base_dtype.newbyteorder("<" if meta["byte_order"] == 0 else ">")
    else:
        dtype = base_dtype

    lines = int(meta["lines"])
    samples = int(meta["samples"])
    bands = int(meta["bands"])

    interleave = meta["interleave"]

    if interleave == "bip":
        shape = (lines, samples, bands)
        mm = np.memmap(
            data_path,
            dtype=dtype,
            mode="r",
            offset=int(meta["header_offset"]),
            shape=shape,
            order="C"
        )
        cube = mm

    elif interleave == "bil":
        shape = (lines, bands, samples)
        mm = np.memmap(
            data_path,
            dtype=dtype,
            mode="r",
            offset=int(meta["header_offset"]),
            shape=shape,
            order="C"
        )
        cube = mm.transpose(0, 2, 1)

    elif interleave == "bsq":
        shape = (bands, lines, samples)
        mm = np.memmap(
            data_path,
            dtype=dtype,
            mode="r",
            offset=int(meta["header_offset"]),
            shape=shape,
            order="C"
        )
        cube = mm.transpose(1, 2, 0)

    else:
        raise RuntimeError(f"Interleave no soportado: {interleave}")

    if len(meta["wavelengths"]) != bands:
        raise RuntimeError(
            f"{header_path}: bands={bands}, wavelengths={len(meta['wavelengths'])}"
        )

    return cube, meta


def otsu_threshold(values, bins=256):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan

    vmin = np.min(values)
    vmax = np.max(values)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return vmin

    hist, edges = np.histogram(values, bins=bins, range=(vmin, vmax))
    hist = hist.astype(float)

    centers = (edges[:-1] + edges[1:]) / 2
    prob = hist / max(hist.sum(), 1)

    omega = np.cumsum(prob)
    mu = np.cumsum(prob * centers)
    mu_t = mu[-1]

    denom = omega * (1 - omega)
    sigma_b = np.zeros_like(denom)

    valid = denom > 0
    sigma_b[valid] = (
        (mu_t * omega[valid] - mu[valid]) ** 2
        / denom[valid]
    )

    idx = int(np.nanargmax(sigma_b))
    return float(centers[idx])


def nearest_band_indices(wavelengths, targets=(550, 700, 850)):
    wavelengths = np.asarray(wavelengths, dtype=float)
    return [
        int(np.argmin(np.abs(wavelengths - t)))
        for t in targets
    ]


def largest_component(mask):
    labels, n = ndimage.label(mask)

    if n == 0:
        return mask

    sizes = ndimage.sum(
        mask,
        labels,
        index=np.arange(1, n + 1)
    )

    largest = int(np.argmax(sizes)) + 1
    return labels == largest


def make_fruit_mask(cube, wavelengths):
    idx = nearest_band_indices(wavelengths, targets=(550, 700, 850))

    proxy = np.nanmedian(
        np.asarray(cube[:, :, idx], dtype=np.float32),
        axis=2
    )

    finite = np.isfinite(proxy)

    if not finite.any():
        return np.zeros(proxy.shape, dtype=bool), {
            "threshold": np.nan,
            "coverage": 0.0,
            "qc_flag": "no_finite_pixels"
        }

    finite_values = proxy[finite]

    threshold = otsu_threshold(finite_values)

    # Probar ambos lados del umbral y escoger el candidato
    # cuya componente mayor sea más plausible como objeto central.
    candidates = []

    for direction in ["high", "low"]:

        raw = (
            (proxy > threshold)
            if direction == "high"
            else (proxy < threshold)
        ) & finite

        raw = ndimage.binary_opening(raw, iterations=1)
        raw = ndimage.binary_closing(raw, iterations=2)
        raw = ndimage.binary_fill_holes(raw)
        raw = largest_component(raw)

        coverage = raw.mean()

        if raw.any():
            yy, xx = np.where(raw)
            cy = yy.mean() / max(raw.shape[0] - 1, 1)
            cx = xx.mean() / max(raw.shape[1] - 1, 1)
            center_distance = math.sqrt((cy - 0.5)**2 + (cx - 0.5)**2)
        else:
            center_distance = 10.0

        # Cobertura preferida amplia: 5%-90%.
        coverage_penalty = 0.0
        if coverage < 0.05:
            coverage_penalty += (0.05 - coverage) * 10
        if coverage > 0.90:
            coverage_penalty += (coverage - 0.90) * 10

        score = center_distance + coverage_penalty

        candidates.append(
            (score, direction, raw, coverage)
        )

    candidates.sort(key=lambda x: x[0])
    _, direction, mask, coverage = candidates[0]

    qc_flag = "ok"
    if coverage < 0.03:
        qc_flag = "mask_too_small"
    elif coverage > 0.95:
        qc_flag = "mask_too_large"

    return mask.astype(bool), {
        "threshold": float(threshold),
        "coverage": float(coverage),
        "direction": direction,
        "qc_flag": qc_flag,
        "proxy_band_indices": idx,
    }


def representative_spectrum(
    cube,
    mask,
    max_pixels=30000,
    rng=None
):
    coords = np.flatnonzero(mask.ravel())

    if coords.size == 0:
        raise RuntimeError("Máscara sin píxeles.")

    if rng is None:
        rng = np.random.default_rng(0)

    if coords.size > max_pixels:
        coords = rng.choice(
            coords,
            size=max_pixels,
            replace=False
        )

    # Materializar solo los píxeles seleccionados.
    cube_arr = np.asarray(cube)
    pixels = cube_arr.reshape(-1, cube_arr.shape[2])[coords]
    pixels = np.asarray(pixels, dtype=np.float32)

    spectrum = np.nanmedian(pixels, axis=0)

    return spectrum, {
        "n_mask_pixels": int(mask.sum()),
        "n_pixels_used": int(len(coords)),
        "spectrum_min": float(np.nanmin(spectrum)),
        "spectrum_max": float(np.nanmax(spectrum)),
        "spectrum_mean": float(np.nanmean(spectrum)),
    }


## 6. Verificación visual de máscaras en ejemplos


In [ ]:
diagnostic_rows = []

for camera in sorted(master["camera_type"].unique()):
    subset = master[master["camera_type"] == camera].copy()

    # Seleccionar un ejemplo aproximadamente intermedio
    row = subset.iloc[len(subset) // 2]

    cube, meta = open_envi_cube(
        row["header_abs"],
        row["data_abs"]
    )

    mask, mask_meta = make_fruit_mask(
        cube,
        meta["wavelengths"]
    )

    idx = nearest_band_indices(
        meta["wavelengths"],
        targets=(550, 700, 850)
    )

    proxy = np.nanmedian(
        np.asarray(cube[:, :, idx], dtype=np.float32),
        axis=2
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(proxy)
    ax.contour(mask.astype(float), levels=[0.5], linewidths=1)
    ax.set_title(
        f"{camera} | fruto {row['fruit_id']} | "
        f"día {row['storage_days_model']} | {row['side']}"
    )
    ax.axis("off")
    fig.tight_layout()

    out_path = (
        FIGURES /
        f"02_mask_diagnostic_{camera}.png"
    )

    fig.savefig(
        out_path,
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()

    diagnostic_rows.append({
        "camera_type": camera,
        "fruit_id": int(row["fruit_id"]),
        "storage_days": int(row["storage_days_model"]),
        "side": row["side"],
        "mask_coverage": mask_meta["coverage"],
        "mask_qc": mask_meta["qc_flag"],
        "figure": str(out_path),
    })

mask_diagnostics = pd.DataFrame(diagnostic_rows)
display(mask_diagnostics)
mask_diagnostics.to_csv(
    TABLES / "06_mask_diagnostic_examples.csv",
    index=False
)


# 7. Extraer espectro mediano de los 793 cubos

La primera ejecución puede tardar varios minutos porque debe leer ~3 GB descomprimidos.

Se crea un **cache por cámara**. Si vuelve a ejecutar el notebook, la extracción se reutiliza.

La extracción genera:

- metadatos de máscara;
- espectro mediano del fruto;
- longitudes de onda exactas;
- señales de QC.

**No se descarta ningún cubo automáticamente.**


In [ ]:
def wavelength_col(w):
    return f"wl_{float(w):.3f}"

def extract_camera_spectra(camera, force=False):
    cache_path = INTERIM / f"kaki_cube_spectra_{camera}.csv"

    if cache_path.exists() and not force:
        cached = pd.read_csv(cache_path)

        if len(cached) == int((master["camera_type"] == camera).sum()):
            print(f"✅ Cache reutilizado: {cache_path}")
            return cached

        print("Cache incompatible; se recalculará.")

    subset = (
        master[master["camera_type"] == camera]
        .sort_values(
            ["storage_days_model", "fruit_id", "side"]
        )
        .copy()
    )

    output_rows = []

    expected_wavelengths = None

    for _, row in tqdm(
        subset.iterrows(),
        total=len(subset),
        desc=f"Extrayendo {camera}"
    ):
        cube, meta = open_envi_cube(
            row["header_abs"],
            row["data_abs"]
        )

        wavelengths = meta["wavelengths"]

        if expected_wavelengths is None:
            expected_wavelengths = wavelengths.copy()
        else:
            if (
                len(wavelengths) != len(expected_wavelengths)
                or not np.allclose(
                    wavelengths,
                    expected_wavelengths,
                    atol=1e-6,
                    rtol=0
                )
            ):
                raise RuntimeError(
                    f"La grilla espectral cambia dentro de {camera}."
                )

        mask, mask_meta = make_fruit_mask(
            cube,
            wavelengths
        )

        try:
            spectrum, spec_meta = representative_spectrum(
                cube,
                mask,
                max_pixels=30000,
                rng=rng
            )
            extraction_error = None
        except Exception as exc:
            spectrum = np.full(len(wavelengths), np.nan)
            spec_meta = {
                "n_mask_pixels": int(mask.sum()),
                "n_pixels_used": 0,
                "spectrum_min": np.nan,
                "spectrum_max": np.nan,
                "spectrum_mean": np.nan,
            }
            extraction_error = repr(exc)

        out = {
            "record_key": row.get("record_key"),
            "fruit_id": int(row["fruit_id"]),
            "storage_days_model": int(row["storage_days_model"]),
            "day_code": row["day_code"],
            "camera_type": camera,
            "side": row["side"],
            "header_file": row["header_file"],
            "data_file": row["data_file"],
            "mask_coverage": mask_meta["coverage"],
            "mask_direction": mask_meta.get("direction"),
            "mask_qc_flag": mask_meta["qc_flag"],
            "mask_threshold": mask_meta["threshold"],
            "n_mask_pixels": spec_meta["n_mask_pixels"],
            "n_pixels_used": spec_meta["n_pixels_used"],
            "spectrum_min": spec_meta["spectrum_min"],
            "spectrum_max": spec_meta["spectrum_max"],
            "spectrum_mean": spec_meta["spectrum_mean"],
            "extraction_error": extraction_error,
        }

        for w, value in zip(wavelengths, spectrum):
            out[wavelength_col(w)] = float(value)

        output_rows.append(out)

    out_df = pd.DataFrame(output_rows)

    out_df.to_csv(
        cache_path,
        index=False
    )

    print("✅ Guardado:", cache_path)
    return out_df


cube_spectra = {}

for camera in sorted(master["camera_type"].unique()):
    cube_spectra[camera] = extract_camera_spectra(
        camera,
        force=False
    )

for camera, df in cube_spectra.items():
    print(camera, df.shape)


## 8. QC de máscaras y espectros


In [ ]:
qc_rows = []

for camera, df in cube_spectra.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]

    qc_rows.append({
        "camera_type": camera,
        "n_cubes": len(df),
        "n_wavelengths": len(wl_cols),
        "mask_coverage_min": df["mask_coverage"].min(),
        "mask_coverage_median": df["mask_coverage"].median(),
        "mask_coverage_max": df["mask_coverage"].max(),
        "n_mask_qc_not_ok": int((df["mask_qc_flag"] != "ok").sum()),
        "n_extraction_errors": int(df["extraction_error"].notna().sum()),
        "n_any_nan_spectrum": int(df[wl_cols].isna().any(axis=1).sum()),
    })

qc_summary = pd.DataFrame(qc_rows)

display(qc_summary)

qc_summary.to_csv(
    TABLES / "07_spectral_extraction_qc_summary.csv",
    index=False
)

for camera, df in cube_spectra.items():
    problematic = df[
        (df["mask_qc_flag"] != "ok")
        | df["extraction_error"].notna()
        | df[[c for c in df.columns if c.startswith("wl_")]].isna().any(axis=1)
    ].copy()

    problematic.to_csv(
        TABLES / f"08_problematic_cubes_{camera}.csv",
        index=False
    )

    print(
        camera,
        "problemáticos:",
        len(problematic)
    )


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))

for camera, df in cube_spectra.items():
    ax.hist(
        df["mask_coverage"].dropna(),
        bins=30,
        alpha=0.5,
        label=camera
    )

ax.set_xlabel("Proporción de píxeles incluidos en la máscara")
ax.set_ylabel("Número de cubos")
ax.set_title("Distribución de cobertura de máscara")
ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
fig.savefig(
    FIGURES / "03_mask_coverage_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# 9. Agregación a nivel fruto×día×cámara

Para evitar pseudo-replicación en la EDA principal:

1. se obtiene un espectro por cubo;
2. `front/back` se agregan mediante **mediana**;
3. la unidad resultante es **un fruto en un día, medido por una cámara**.

Esperamos:

\[
199\ 	ext{fruit-days}	imes2\ 	ext{cámaras}=398
\]

filas agregadas.


In [ ]:
fruit_day_camera = {}

for camera, df in cube_spectra.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]

    agg = (
        df.groupby(
            ["fruit_id", "storage_days_model", "day_code", "camera_type"],
            as_index=False
        )
        .agg(
            {
                **{c: "median" for c in wl_cols},
                "side": "nunique",
                "mask_coverage": "median",
            }
        )
        .rename(
            columns={
                "side": "n_sides_aggregated",
                "mask_coverage": "median_mask_coverage",
            }
        )
    )

    fruit_day_camera[camera] = agg

    out_path = (
        INTERIM /
        f"kaki_fruit_day_spectra_{camera}.csv"
    )

    agg.to_csv(out_path, index=False)

    print(
        camera,
        "fruit-day rows:",
        len(agg),
        "| n_sides:",
        agg["n_sides_aggregated"].value_counts().sort_index().to_dict()
    )

assert sum(len(x) for x in fruit_day_camera.values()) == 398


## 10. Grillas espectrales por cámara


In [ ]:
camera_grid_rows = []

for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]
    wavelengths = np.array(
        [float(c.replace("wl_", "")) for c in wl_cols],
        dtype=float
    )

    camera_grid_rows.append({
        "camera_type": camera,
        "n_wavelengths": len(wavelengths),
        "wavelength_min_nm": float(wavelengths.min()),
        "wavelength_max_nm": float(wavelengths.max()),
        "median_spacing_nm": float(np.median(np.diff(wavelengths))),
    })

camera_grid = pd.DataFrame(camera_grid_rows)
display(camera_grid)

camera_grid.to_csv(
    TABLES / "09_camera_spectral_grids.csv",
    index=False
)


# 11. Espectro medio por día de almacenamiento — cámaras separadas

Esta figura es descriptiva.  
No interpreta causalmente diferencias día a día porque `storage_days` coincide con sesiones de adquisición.


In [ ]:
for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]
    wavelengths = np.array(
        [float(c.replace("wl_", "")) for c in wl_cols]
    )

    fig, ax = plt.subplots(figsize=(10, 5.5))

    for storage_day, g in df.groupby("storage_days_model"):
        mean_spectrum = g[wl_cols].mean(axis=0).to_numpy(dtype=float)

        ax.plot(
            wavelengths,
            mean_spectrum,
            label=f"Día {int(storage_day)}"
        )

    ax.set_xlabel("Longitud de onda (nm)")
    ax.set_ylabel("Señal espectral representativa")
    ax.set_title(f"{camera}: espectro medio por día de almacenamiento")
    ax.legend(ncol=2, fontsize=8)
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / f"04_mean_spectra_by_day_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


# 12. Cambio longitudinal dentro del mismo fruto respecto al día 0

Este análisis es central.

Para cada fruto y cámara:

\[
\Delta R_{i,t}(\lambda)=R_{i,t}(\lambda)-R_{i,0}(\lambda)
\]

Así se reduce el peso de las diferencias basales entre frutos y se observa la dirección del cambio **dentro del mismo fruto**.


In [ ]:
delta_tables = {}

for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]

    baseline = (
        df[df["storage_days_model"] == 0]
        .set_index("fruit_id")[wl_cols]
    )

    delta_rows = []

    for _, row in df.iterrows():
        fruit_id = int(row["fruit_id"])

        if fruit_id not in baseline.index:
            continue

        spectrum = row[wl_cols].to_numpy(dtype=float)
        base = baseline.loc[fruit_id].to_numpy(dtype=float)

        delta = spectrum - base

        out = {
            "fruit_id": fruit_id,
            "storage_days_model": int(row["storage_days_model"]),
            "camera_type": camera,
        }

        for col, value in zip(wl_cols, delta):
            out[col] = float(value)

        delta_rows.append(out)

    delta_df = pd.DataFrame(delta_rows)
    delta_tables[camera] = delta_df

    delta_df.to_csv(
        INTERIM / f"kaki_within_fruit_delta_from_day0_{camera}.csv",
        index=False
    )

    assert len(delta_df) == len(df)

    print(camera, "delta rows:", len(delta_df))


In [ ]:
for camera, delta_df in delta_tables.items():
    wl_cols = [c for c in delta_df.columns if c.startswith("wl_")]
    wavelengths = np.array(
        [float(c.replace("wl_", "")) for c in wl_cols]
    )

    fig, ax = plt.subplots(figsize=(10, 5.5))

    for day, g in delta_df.groupby("storage_days_model"):
        if int(day) == 0:
            continue

        mean_delta = g[wl_cols].mean(axis=0).to_numpy(dtype=float)

        ax.plot(
            wavelengths,
            mean_delta,
            label=f"Día {int(day)}"
        )

    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Longitud de onda (nm)")
    ax.set_ylabel("Δ señal respecto al día 0")
    ax.set_title(
        f"{camera}: cambio longitudinal medio dentro del fruto"
    )
    ax.legend(ncol=2, fontsize=8)
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / f"05_within_fruit_delta_from_day0_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 13. Magnitud del cambio longitudinal

Se calcula la distancia RMS espectral entre cada día y el día 0 del **mismo fruto**:

\[
D_{i,t}=
\sqrt{
rac{1}{p}\sum_{\lambda}
(R_{i,t,\lambda}-R_{i,0,\lambda})^2}
\]

No es una métrica de desempeño; es una medida descriptiva de cuánto se desplaza el espectro respecto al estado inicial.


In [ ]:
longitudinal_distance_rows = []

for camera, delta_df in delta_tables.items():
    wl_cols = [c for c in delta_df.columns if c.startswith("wl_")]

    X = delta_df[wl_cols].to_numpy(dtype=float)

    rms = np.sqrt(
        np.nanmean(X**2, axis=1)
    )

    tmp = delta_df[
        ["fruit_id", "storage_days_model", "camera_type"]
    ].copy()

    tmp["rms_delta_from_day0"] = rms

    longitudinal_distance_rows.append(tmp)

longitudinal_distance = pd.concat(
    longitudinal_distance_rows,
    ignore_index=True
)

display(
    longitudinal_distance.groupby(
        ["camera_type", "storage_days_model"]
    )["rms_delta_from_day0"]
    .agg(["count", "median", "mean", "std"])
)

longitudinal_distance.to_csv(
    TABLES / "10_longitudinal_rms_distance_from_day0.csv",
    index=False
)


In [ ]:
for camera, gcam in longitudinal_distance.groupby("camera_type"):

    summary = (
        gcam.groupby("storage_days_model")["rms_delta_from_day0"]
        .agg(["median", "mean", "std"])
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(8, 4.8))

    ax.plot(
        summary["storage_days_model"],
        summary["median"],
        marker="o",
        label="Mediana"
    )

    ax.plot(
        summary["storage_days_model"],
        summary["mean"],
        marker="s",
        label="Media"
    )

    ax.set_xlabel("Días de almacenamiento")
    ax.set_ylabel("Distancia RMS respecto al día 0")
    ax.set_title(f"{camera}: magnitud del cambio longitudinal")
    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / f"06_longitudinal_distance_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


# 14. Concordancia front/back dentro del mismo fruto-día

La diferencia entre lados es una fuente de variabilidad técnica/espacial.

Se calcula el RMSE espectral entre front y back para cada fruto-día-cámara cuando existen ambas vistas.


In [ ]:
side_difference_rows = []

for camera, df in cube_spectra.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]

    for (fruit_id, day), g in df.groupby(
        ["fruit_id", "storage_days_model"]
    ):
        if set(g["side"].dropna().astype(str)) < {"front", "back"}:
            continue

        front = (
            g[g["side"] == "front"]
            .iloc[0][wl_cols]
            .to_numpy(dtype=float)
        )

        back = (
            g[g["side"] == "back"]
            .iloc[0][wl_cols]
            .to_numpy(dtype=float)
        )

        rmse = float(
            np.sqrt(
                np.nanmean(
                    (front - back)**2
                )
            )
        )

        side_difference_rows.append({
            "camera_type": camera,
            "fruit_id": int(fruit_id),
            "storage_days_model": int(day),
            "front_back_rmse": rmse,
        })

side_difference = pd.DataFrame(side_difference_rows)

display(
    side_difference.groupby("camera_type")["front_back_rmse"]
    .agg(["count", "median", "mean", "std", "max"])
)

side_difference.to_csv(
    TABLES / "11_front_back_spectral_difference.csv",
    index=False
)


# 15. PCA longitudinal por cámara — espectros agregados fruit-day

Se realizan dos PCA descriptivos:

1. **RAW-centered**: solo centrado por longitud de onda.
2. **SNV**: normalización por espectro antes del PCA.

SNV aquí se usa solo como herramienta exploratoria, no como decisión final de preprocesamiento.


In [ ]:
def snv_rows(X):
    X = np.asarray(X, dtype=float)
    mu = np.nanmean(X, axis=1, keepdims=True)
    sd = np.nanstd(X, axis=1, keepdims=True)
    sd[sd == 0] = 1.0
    return (X - mu) / sd


pca_summary_rows = []

for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]
    X_raw = df[wl_cols].to_numpy(dtype=float)

    for mode in ["raw_centered", "snv"]:
        X = X_raw.copy()

        if mode == "snv":
            X = snv_rows(X)

        # PCA sklearn centra automáticamente; no escalamos cada banda.
        pca = PCA(n_components=min(10, X.shape[0], X.shape[1]))
        scores = pca.fit_transform(X)

        score_df = df[
            ["fruit_id", "storage_days_model", "camera_type"]
        ].copy()

        score_df["PC1"] = scores[:, 0]
        score_df["PC2"] = scores[:, 1]

        score_df.to_csv(
            TABLES / f"12_pca_scores_{camera}_{mode}.csv",
            index=False
        )

        pca_summary_rows.append({
            "camera_type": camera,
            "mode": mode,
            "pc1_explained": float(pca.explained_variance_ratio_[0]),
            "pc2_explained": float(pca.explained_variance_ratio_[1]),
            "pc1_pc2_cumulative": float(
                pca.explained_variance_ratio_[:2].sum()
            ),
        })

        fig, ax = plt.subplots(figsize=(7, 5.5))

        scatter = ax.scatter(
            scores[:, 0],
            scores[:, 1],
            c=df["storage_days_model"].to_numpy(),
            s=28,
            alpha=0.8
        )

        ax.set_xlabel(
            f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)"
        )
        ax.set_ylabel(
            f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"
        )
        ax.set_title(
            f"{camera} — PCA {mode} a nivel fruto-día"
        )

        cb = fig.colorbar(scatter, ax=ax)
        cb.set_label("Storage days")

        ax.grid(alpha=0.15)
        fig.tight_layout()

        fig.savefig(
            FIGURES / f"07_pca_{camera}_{mode}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

pca_summary = pd.DataFrame(pca_summary_rows)
display(pca_summary)

pca_summary.to_csv(
    TABLES / "13_pca_explained_variance.csv",
    index=False
)


# 16. Asociación espectral con `storage_days` — descriptiva

Se calcula Spearman por longitud de onda en el nivel **fruit-day**.

Estos coeficientes **no se interpretarán como evidencia inferencial independiente**, porque existen medidas repetidas y confusión día–sesión. Se utilizan únicamente para localizar regiones espectrales con tendencia monotónica.


In [ ]:
association_tables = {}

for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]
    y = df["storage_days_model"].to_numpy(dtype=float)

    rows = []

    for col in wl_cols:
        x = df[col].to_numpy(dtype=float)
        rho, p = spearmanr(x, y, nan_policy="omit")

        rows.append({
            "camera_type": camera,
            "wavelength_nm": float(col.replace("wl_", "")),
            "spearman_rho": float(rho),
            "descriptive_pvalue": float(p),
        })

    assoc = pd.DataFrame(rows)
    association_tables[camera] = assoc

    assoc.to_csv(
        TABLES / f"14_spearman_storage_days_{camera}.csv",
        index=False
    )

    top = (
        assoc.assign(abs_rho=assoc["spearman_rho"].abs())
        .sort_values("abs_rho", ascending=False)
        .head(15)
    )

    print("\n", camera, "— mayores |rho|")
    display(top)

    fig, ax = plt.subplots(figsize=(10, 4.8))

    ax.plot(
        assoc["wavelength_nm"],
        assoc["spearman_rho"]
    )

    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Longitud de onda (nm)")
    ax.set_ylabel("Spearman ρ")
    ax.set_title(
        f"{camera}: asociación descriptiva longitud de onda × storage_days"
    )
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / f"08_spearman_by_wavelength_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


# 17. Variabilidad entre frutos vs. dentro del fruto

Para cada longitud de onda se calcula de forma descriptiva:

- varianza de los promedios por fruto = componente **entre frutos**;
- media de la varianza temporal dentro de cada fruto = componente **intra-fruto**.

No es un modelo mixto formal; sirve para localizar regiones donde la señal longitudinal es grande respecto a la heterogeneidad basal.


In [ ]:
variance_component_tables = {}

for camera, df in fruit_day_camera.items():
    wl_cols = [c for c in df.columns if c.startswith("wl_")]

    fruit_means = (
        df.groupby("fruit_id")[wl_cols]
        .mean()
    )

    between_var = fruit_means.var(axis=0, ddof=1)

    within_by_fruit = []
    for _, g in df.groupby("fruit_id"):
        if len(g) < 2:
            continue
        within_by_fruit.append(
            g[wl_cols].var(axis=0, ddof=1)
        )

    within_var = pd.DataFrame(within_by_fruit).mean(axis=0)

    eps = 1e-12

    ratio = (
        within_var /
        (between_var + within_var + eps)
    )

    wavelengths = np.array(
        [float(c.replace("wl_", "")) for c in wl_cols]
    )

    vc = pd.DataFrame({
        "camera_type": camera,
        "wavelength_nm": wavelengths,
        "between_fruit_variance": between_var.to_numpy(dtype=float),
        "within_fruit_temporal_variance": within_var.to_numpy(dtype=float),
        "within_fraction_of_total_descriptive": ratio.to_numpy(dtype=float),
    })

    variance_component_tables[camera] = vc

    vc.to_csv(
        TABLES / f"15_variance_components_{camera}.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(10, 4.8))

    ax.plot(
        vc["wavelength_nm"],
        vc["within_fraction_of_total_descriptive"]
    )

    ax.set_xlabel("Longitud de onda (nm)")
    ax.set_ylabel("Fracción descriptiva intra-fruto")
    ax.set_ylim(0, 1)
    ax.set_title(
        f"{camera}: variabilidad temporal intra-fruto vs. entre frutos"
    )
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / f"09_within_vs_between_variability_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


# 18. Comparación temporal entre cámaras en el rango espectral común

No se comparan directamente valores absolutos de reflectancia entre sensores.

Se comparan las **curvas de cambio relativo respecto al día 0**, interpoladas a una grilla común de 5 nm dentro del solapamiento espectral.


In [ ]:
camera_names = sorted(fruit_day_camera.keys())

if len(camera_names) != 2:
    print(
        "Se esperaban dos cámaras; se omite comparación cruzada."
    )
    cross_camera_summary = pd.DataFrame()

else:
    cam_a, cam_b = camera_names

    def get_wavelengths(df):
        return np.array(
            [
                float(c.replace("wl_", ""))
                for c in df.columns
                if c.startswith("wl_")
            ],
            dtype=float
        )

    wa = get_wavelengths(delta_tables[cam_a])
    wb = get_wavelengths(delta_tables[cam_b])

    common_min = max(wa.min(), wb.min())
    common_max = min(wa.max(), wb.max())

    common_grid = np.arange(
        math.ceil(common_min / 5) * 5,
        math.floor(common_max / 5) * 5 + 0.1,
        5.0
    )

    comparison_rows = []

    for day in sorted(
        set(delta_tables[cam_a]["storage_days_model"])
        & set(delta_tables[cam_b]["storage_days_model"])
    ):
        if int(day) == 0:
            continue

        ga = delta_tables[cam_a][
            delta_tables[cam_a]["storage_days_model"] == day
        ]
        gb = delta_tables[cam_b][
            delta_tables[cam_b]["storage_days_model"] == day
        ]

        cols_a = [c for c in ga.columns if c.startswith("wl_")]
        cols_b = [c for c in gb.columns if c.startswith("wl_")]

        mean_a = ga[cols_a].mean(axis=0).to_numpy(dtype=float)
        mean_b = gb[cols_b].mean(axis=0).to_numpy(dtype=float)

        interp_a = np.interp(common_grid, wa, mean_a)
        interp_b = np.interp(common_grid, wb, mean_b)

        # Estandarizar cada curva para comparar forma, no amplitud absoluta
        def z(v):
            sd = np.std(v)
            return (v - np.mean(v)) / (sd if sd > 0 else 1)

        za = z(interp_a)
        zb = z(interp_b)

        rho, _ = spearmanr(za, zb)

        comparison_rows.append({
            "storage_days_model": int(day),
            "camera_a": cam_a,
            "camera_b": cam_b,
            "common_min_nm": float(common_grid.min()),
            "common_max_nm": float(common_grid.max()),
            "n_common_grid_points": int(len(common_grid)),
            "spearman_shape_agreement": float(rho),
        })

    cross_camera_summary = pd.DataFrame(comparison_rows)

    display(cross_camera_summary)

    cross_camera_summary.to_csv(
        TABLES / "16_cross_camera_temporal_shape_agreement.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(8, 4.8))

    ax.plot(
        cross_camera_summary["storage_days_model"],
        cross_camera_summary["spearman_shape_agreement"],
        marker="o"
    )

    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Días de almacenamiento")
    ax.set_ylabel("Spearman ρ entre patrones temporales")
    ax.set_title(
        "Concordancia entre cámaras del patrón espectral de cambio"
    )
    ax.set_ylim(-1.05, 1.05)
    ax.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(
        FIGURES / "10_cross_camera_temporal_agreement.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


# 19. Ground truth destructivo terminal: firmeza y estado de madurez

Las mediciones destructivas pertenecen al **día terminal del fruto**.  
Se colapsan duplicados de vistas/cámaras para obtener una fila por fruto.

No se propaga la firmeza terminal hacia días anteriores.


In [ ]:
# Seleccionar solo registros con anotación destructiva original disponible.
# "storage_days_original" fue conservado en el índice corregido.
if "storage_days_original" not in master.columns:
    print(
        "⚠️ storage_days_original no está presente; "
        "se omite análisis de endpoint destructivo."
    )
    endpoint_fruit = pd.DataFrame()

else:
    endpoint = master[
        pd.to_numeric(
            master["storage_days_original"],
            errors="coerce"
        ).notna()
    ].copy()

    candidate_cols = [
        "fruit_id",
        "storage_days_model",
        "firmness",
        "ripeness_state",
        "ripeness_state_fine",
        "init_weight",
        "end_weight",
    ]

    for c in candidate_cols:
        if c not in endpoint.columns:
            endpoint[c] = np.nan

    endpoint_fruit = (
        endpoint[candidate_cols]
        .sort_values(["fruit_id", "storage_days_model"])
        .drop_duplicates("fruit_id")
        .copy()
    )

    endpoint_fruit.to_csv(
        TABLES / "17_terminal_ground_truth_by_fruit.csv",
        index=False
    )

    print("Frutos con endpoint:", len(endpoint_fruit))
    display(endpoint_fruit.head(10))

    endpoint_fruit["firmness_numeric"] = pd.to_numeric(
        endpoint_fruit["firmness"],
        errors="coerce"
    )

    valid = endpoint_fruit[
        endpoint_fruit["firmness_numeric"].notna()
    ]

    if len(valid) >= 5:
        rho, p = spearmanr(
            valid["storage_days_model"],
            valid["firmness_numeric"]
        )

        print(
            f"Spearman terminal day × firmness: "
            f"rho={rho:.3f}, p descriptivo={p:.4g}"
        )

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.scatter(
            valid["storage_days_model"],
            valid["firmness_numeric"],
            s=45,
            alpha=0.8
        )

        ax.set_xlabel("Día terminal de almacenamiento")
        ax.set_ylabel("Firmeza terminal")
        ax.set_title("Firmeza destructiva vs. día terminal")
        ax.grid(alpha=0.2)

        fig.tight_layout()
        fig.savefig(
            FIGURES / "11_terminal_firmness_vs_terminal_day.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()


# 20. Resumen automático del EDA

El JSON conserva hechos descriptivos y alertas metodológicas.  
No contiene conclusiones predictivas.


In [ ]:
def py(obj):
    if isinstance(obj, dict):
        return {str(k): py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [py(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return py(obj.tolist())
    return obj


summary = {
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK_ID,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "unit_structure": {
        "physical_fruits": int(master["fruit_id"].nunique()),
        "fruit_days": int(
            master[["fruit_id", "storage_days_model"]]
            .drop_duplicates()
            .shape[0]
        ),
        "physical_hsi_cubes": int(len(master)),
        "storage_days": sorted(
            [int(x) for x in master["storage_days_model"].unique()]
        ),
        "cameras": {
            str(k): int(v)
            for k, v in master["camera_type"].value_counts().to_dict().items()
        },
    },
    "spectral_extraction": {
        row["camera_type"]: {
            "n_cubes": int(row["n_cubes"]),
            "n_wavelengths": int(row["n_wavelengths"]),
            "n_mask_qc_not_ok": int(row["n_mask_qc_not_ok"]),
            "n_extraction_errors": int(row["n_extraction_errors"]),
            "median_mask_coverage": float(row["mask_coverage_median"]),
        }
        for _, row in qc_summary.iterrows()
    },
    "fruit_day_camera_rows": {
        camera: int(len(df))
        for camera, df in fruit_day_camera.items()
    },
    "methodological_alerts": [
        "Repeated measures from the same physical fruit are not independent.",
        "Front/back views are aggregated for the principal EDA.",
        "The two cameras are analyzed separately before cross-camera comparison.",
        "Storage day is linked to acquisition session/day_code in this measurement series.",
        "Fruit count decreases over storage time because fruits leave the longitudinal sequence at destructive endpoint.",
        "Terminal firmness/ripeness measurements are not propagated backward to earlier days.",
        "No predictive model or train/test split is created in this notebook."
    ]
}

summary_path = RESULTS / "kaki_eda_longitudinal_summary.json"

summary_path.write_text(
    json.dumps(
        py(summary),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(py(summary), indent=2, ensure_ascii=False))


## 21. Notas automáticas para revisión científica


In [ ]:
review_notes = []

review_notes.append(
    f"Dataset analyzed: {master['fruit_id'].nunique()} physical fruits, "
    f"{master[['fruit_id','storage_days_model']].drop_duplicates().shape[0]} fruit-days, "
    f"{len(master)} HSI cubes."
)

review_notes.append(
    "Main EDA unit: fruit-day-camera after median aggregation of front/back."
)

for _, row in qc_summary.iterrows():
    review_notes.append(
        f"{row['camera_type']}: {int(row['n_wavelengths'])} wavelengths, "
        f"{int(row['n_mask_qc_not_ok'])} mask-QC flags, "
        f"{int(row['n_extraction_errors'])} extraction errors."
    )

review_notes.append(
    "Storage-day/session confounding remains a critical limitation: "
    "a storage day corresponds to an acquisition session in this series."
)

review_notes.append(
    "Progressive destructive endpoints generate an unbalanced longitudinal panel: "
    "late storage days contain only the fruits that remained until those endpoints."
)

review_notes.append(
    "No automatic observation deletion was performed from spectral QC flags."
)

review_notes.append(
    "No predictive modeling was performed."
)

notes_path = RESULTS / "EDA_REVIEW_NOTES.txt"

notes_path.write_text(
    "\n".join(f"- {x}" for x in review_notes) + "\n",
    encoding="utf-8"
)

print(notes_path.read_text())
